# 02 — From local interpolation to global learning

We test two progressively stronger structural hypotheses using **exactly the same 40 training states and 8 validation states**:

\[
\text{local thermodynamic proximity}\;\longrightarrow\;\text{global linear response}.
\]

The point is not only which model has the smaller error. The important question is **what scientific assumption changes**.

In [ ]:
#@title 0. Workshop setup — run once { display-mode: "form" }
# This cell intentionally hides infrastructure so workshop time stays focused on physics.

from pathlib import Path
import hashlib, importlib.util, os, shutil, subprocess, sys, urllib.request, zipfile

ASSET_URL = "" #@param {type:"string"}
EXPECTED_ASSET_SHA256 = "2e75fd65ad39a9dec41f7b089c2aafe51a6bb14eeee049b055be8ea3953a7bea"
WORKSHOP_ROOT = Path("/content/ThermoRDF-Workshop")
ASSET_NAME = "ThermoRDF-Colab-Assets.zip"

# Local/instructor execution override used only for automated testing.
_local_root = os.environ.get("THERMORDF_WORKSHOP_ROOT", "").strip()
if _local_root:
    WORKSHOP_ROOT = Path(_local_root).resolve()
else:
    ready = (WORKSHOP_ROOT / "data/teaching/stage_02_b48_train_40.csv.gz").is_file()
    if not ready:
        archive = Path("/content") / ASSET_NAME
        if ASSET_URL.strip():
            print("Downloading workshop assets ...")
            urllib.request.urlretrieve(ASSET_URL.strip(), archive)
        else:
            try:
                from google.colab import files
            except ImportError as exc:
                raise RuntimeError("This notebook is configured for Google Colab. Set THERMORDF_WORKSHOP_ROOT for local testing.") from exc
            print(f"Upload the companion file: {ASSET_NAME}")
            uploaded = files.upload()
            if ASSET_NAME not in uploaded:
                raise RuntimeError(f"Expected {ASSET_NAME}. Please rerun this cell and upload that file.")
            archive.write_bytes(uploaded[ASSET_NAME])

        digest = hashlib.sha256(archive.read_bytes()).hexdigest()
        if digest != EXPECTED_ASSET_SHA256:
            raise RuntimeError("Asset bundle checksum mismatch. Use the bundle distributed with these notebooks.")

        if WORKSHOP_ROOT.exists():
            shutil.rmtree(WORKSHOP_ROOT)
        WORKSHOP_ROOT.mkdir(parents=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(WORKSHOP_ROOT)

_required = {"numpy":"numpy", "pandas":"pandas", "matplotlib":"matplotlib", "scikit-learn":"sklearn", "torch":"torch"}
_missing = [pkg for pkg, module in _required.items() if importlib.util.find_spec(module) is None]
if _missing:
    print("Installing missing Colab packages:", ", ".join(_missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

sys.path.insert(0, str(WORKSHOP_ROOT / "src"))
from thermordf_workshop import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.set_num_threads(min(2, os.cpu_count() or 1))
data = load_workshop_data(WORKSHOP_ROOT)
print(f"Workshop ready | {len(data.train)} training + {len(data.validation)} validation states | {len(data.r_nm)} RDF coordinates")

## Hypothesis 1 — local thermodynamic smoothness

IDW asks whether nearby points in normalised \((T',P')\) space carry the most useful structural information about a target state. The distance-decay exponent is fixed at \(p=2\).

In [ ]:
idw = fit_idw(data)
idw_prediction = idw.predict_frame(data.validation)
idw_result = evaluate_predictions(data, idw_prediction)
print(f"development validation RDF RMSE: {idw_result.attrs['global_rdf_rmse']:.4f}")

### Same demanding reference state

Throughout model development we return to the same validation state, \(T=200\) K and \(P=1\) bar. Keeping the physical state fixed makes changes in reconstruction easy to interpret.

In [ ]:
plot_reference_prediction(
    data, idw_prediction, "T200_Bar0001",
    label="IDW", line_color="#1C778E", with_error=True
);

### Observe

- Where is the disagreement concentrated?
- Is the sharp first-shell structure harder than the smoother outer region?
- What does this tell you about **thermodynamic proximity as a modelling hypothesis**?

## Hypothesis 2 — one global linear structural response

Ridge replaces local interpolation with a single global relation:

\[
\widehat{\mathbf g}=\mathbf b_0 + T'\mathbf b_T + P'\mathbf b_P.
\]

All 40 training states contribute to the same response; regularisation controls the coefficient amplitudes.

In [ ]:
ridge = fit_ridge(data)
ridge_prediction = ridge.predict_frame(data.validation)
ridge_result = evaluate_predictions(data, ridge_prediction)
print(f"selected alpha: {ridge.alpha:g}")
print(f"development validation RDF RMSE: {ridge_result.attrs['global_rdf_rmse']:.4f}")

In [ ]:
plot_reference_prediction(
    data, ridge_prediction, "T200_Bar0001",
    label="Ridge", line_color="#72549A", with_error=True
);

## Compare the hypotheses

The same data now support two different structural assumptions.

In [ ]:
scores = {
    "IDW": idw_result.attrs["global_rdf_rmse"],
    "Ridge": ridge_result.attrs["global_rdf_rmse"],
}
plot_global_comparison(scores);

### Interpret

Ridge improves the development validation result relative to IDW. This supports the existence of an exploitable **global structural trend**, not only local thermodynamic smoothness.

But a linear model imposes the same form of thermodynamic sensitivity throughout the domain.

\[
\boxed{\text{Next question: is the global structural relation strictly linear?}}
\]